We load the df.

In [6]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
df = pd.read_parquet("../data/processed/autoimmune_cleaned.parquet")

Separate target and features.

In [7]:
TARGET = "Diagnosis"
X = df.drop(columns=[TARGET, "Patient_ID"])
y = df[TARGET]

Identify features

In [9]:
# Clinical symptoms (binary)
symptom_cols = [
    "Low-grade fever", "Fatigue or chronic tiredness", "Dizziness",
    "Weight loss", "Rashes and skin lesions", "Stiffness in the joints",
    "Brittle hair or hair loss", "Dry eyes and/or mouth",
    "General 'unwell' feeling", "Joint pain"
]

# Autoantibodies (binary)
autoantibody_cols = [
    "ACPA", "ANA", "ANCA", "ASCA", "ASMA", "C1_inhibitor",
    "DGP", "EMA", "Esbach", "IgG_IgE_receptor",
    "Progesterone_antibodies", "Rheumatoid factor", "pANCA",
    "Anti-SMA", "Anti-Sm", "Anti-TPO", "Anti-Tg", "Anti-dsDNA",
    "Anti_BP180", "Anti_BP230", "Anti_CBir1", "Anti_IF", "Anti_Jo1",
    "Anti_La_SSB", "Anti_Mi2", "Anti_OmpC", "Anti_RNP", "Anti_Ro_SSA",
    "Anti_SRP", "Anti_Sm", "Anti_TIF1", "Anti_desmoglein_1",
    "Anti_desmoglein_3", "Anti_dsDNA", "Anti_enterocyte_antibodies",
    "Anti_epidermal_basement_membrane_IgA", "Anti_parietal_cell",
    "Anti_tTG", "Anti_tissue_transglutaminase", "Anti_type_VII_collagen",
    "anti_LKM1", "anti_Scl_70", "anti_centromere",
]

Gender encoding.

In [12]:
X["Gender"] = X["Gender"].map({"Male": 0, "Female": 1})

Now target.

In [13]:
le = LabelEncoder()
y_encoded = le.fit_transform(y)

label_map = dict(zip(le.classes_, le.transform(le.classes_)))
print(label_map)

{'Acute disseminated encephalomyelitis': np.int64(0), 'Acute motor axonal neuropathy': np.int64(1), "Addison's disease": np.int64(2), 'Alopecia areata': np.int64(3), 'Anti-NMDA receptor encephalitis': np.int64(4), 'Antiphospholipid syndrome': np.int64(5), 'Autoimmune angioedema': np.int64(6), 'Autoimmune encephalitis': np.int64(7), 'Autoimmune enteropathy': np.int64(8), 'Autoimmune hemolytic anemia': np.int64(9), 'Autoimmune hepatitis': np.int64(10), 'Autoimmune oophoritis': np.int64(11), 'Autoimmune orchitis': np.int64(12), 'Autoimmune pancreatitis': np.int64(13), 'Autoimmune polyendocrine syndrome type 1 (APS1)': np.int64(14), 'Autoimmune polyendocrine syndrome type 2 (APS2)': np.int64(15), 'Autoimmune polyendocrine syndrome type 3 (APS3)': np.int64(16), 'Autoimmune progesterone dermatitis': np.int64(17), 'Autoimmune retinopathy': np.int64(18), 'Autoimmune urticaria': np.int64(19), 'Autoimmune uveitis': np.int64(20), 'Balo concentric sclerosis': np.int64(21), "Behçet's disease": np.i

Now, immunological ratios.

In [ ]:
# Neutrophil-to-Lymphocyte Ratio (NLR) - systematic inflamation marker
X["NLR"] = X["Neutrophils"] / (X["Lymphocytes"] + 1e-6)

# Platelet-to-Lymphocyte Ratio (PLR)
X["PLR"] = X["PLT_Count"] / (X["Lymphocytes"] + 1e-6)

# Monocyte-to-Lymphocyte Ratio (MLR)
X["MLR"] = X["Monocytes"] / (X["Lymphocytes"] + 1e-6)

# C3/C4 ratio - disfunción del complemento
X["C3_C4_ratio"] = X["C3"] / (X["C4"] + 1e-6)

# ESR / CRP ratio (useful when one is altered and the other is not)
X["ESR_CRP_ratio"] = X["ESR"] / (X["CRP"] + 1e-6)

# Total positive count (Autoimmune load)
X["autoantibody_count"] = X[autoantibody_cols].sum(axis=1)

# Total symptom count
X["symptom_count"] = X[symptom_cols].sum(axis=1)

# Age × duration (proxy for chronicity)
X["age_x_duration"] = X["Age"] * X["Sickness_Duration_Months"]

Scale numerical features.

In [ ]:
cbc_cols = [
    "RBC_Count", "Hemoglobin", "Hematocrit", "MCV", "MCH", "MCHC", "RDW",
    "Reticulocyte_Count", "WBC_Count", "Neutrophils", "Lymphocytes",
    "Monocytes", "Eosinophils", "Basophils", "PLT_Count", "MPV"
]
inflam_cols = ["ESR", "CRP", "C3", "C4", "MBL_Level"]
demo_cols = ["Age", "Sickness_Duration_Months"]

num_cols_to_scale = cbc_cols + inflam_cols + demo_cols + [
    "NLR", "PLR", "MLR", "C3_C4_ratio", "ESR_CRP_ratio",
    "age_x_duration"
]

scaler = StandardScaler()
X[num_cols_to_scale] = scaler.fit_transform(X[num_cols_to_scale])